# Michigan Traders — Module 9
# Algorithm Framework, Risk Management & Competition Readiness  ·  *ANSWER KEY*

**Series:** MAT Education · ML & Capstone
**Level:** Intermediate (builds on Modules 1–8)
**Format:** **Guided + Your-Turn**, with self-checking exercises you can run on your own laptop.

---

Eight modules have been about finding an edge and not fooling yourself about it. This one is about the two things that stand between a signal and a submission.

**First: structure.** Module 3 built algorithms as one class with everything inside `on_data`. That works for one signal on two stocks. It stops working when you have a universe that changes monthly, three signals that disagree, position sizing rules, and a drawdown limit — at which point `on_data` becomes a thousand lines that nobody can test in pieces. QuantConnect's **Algorithm Framework** splits that into five components with defined interfaces, so each one can be swapped and reasoned about alone.

**Second: risk.** Every performance number you have computed so far assumed you stayed in the trade. Risk management is what decides whether you actually do. It is also the part students most reliably get backwards, so this module measures it instead of asserting it — and the measurements are not what you would guess.

| Technique | Return | Drawdown | Sharpe |
|---|---|---|---|
| Diversification | unchanged | **much lower** | **higher** |
| Position sizing by volatility | **higher** | **lower** | **higher** |
| Volatility targeting | **higher** | **lower** | **higher** |
| Drawdown kill switch | **lower** | **much lower** | slightly lower |
| Trailing stop | *unpredictable* | *unpredictable* | *unpredictable* |

The two everyone reaches for first — stop losses and kill switches — are the two that cost the most and deliver the least. The two nobody finds exciting are the ones that work. You will measure all of them rather than take my word for it.

## How to use this notebook

| Cell type | Where it runs | What to do |
|---|---|---|
| 🟢 **Local cell** | your laptop's Jupyter | Run it. Output is baked in so you can read along. |
| 🔵 **QC cell** | QuantConnect (LEAN) | Copy into an algorithm. It will *not* run locally. |

Answers are in `09_Algorithm_Framework_Risk_and_Competition_SOLUTIONS.ipynb`.

### Setup — five assets, one factor, one crisis

We need a portfolio to manage, so we simulate five assets driven by a common market factor plus their own idiosyncratic noise. Each has a different **beta** to that factor — the sensitivity you measured in Module 6 section 7.

`GOLD` has a *negative* beta, which makes it the diversifier of the group. `TECH` has the highest beta and the highest return. Partway through, the factor enters a 90-day crisis: strongly negative drift and roughly three times the usual volatility.

That crisis is the point. Risk management is invisible until something goes wrong.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 4)

rng = np.random.default_rng(3)
n, crash_start, crash_len = 2016, 1000, 90          # 8 years of business days
dates = pd.bdate_range("2016-01-04", periods=n)

drift = np.full(n, 0.00045)
vol = np.full(n, 0.009)
drift[crash_start:crash_start + crash_len] = -0.0075     # the crisis
vol[crash_start:crash_start + crash_len] = 0.028
factor = rng.normal(drift, vol)

betas = {"MKT_ETF": 1.00, "TECH": 1.35, "BANK": 1.15, "STAPLE": 0.55, "GOLD": -0.25}
idio = {"MKT_ETF": 0.002, "TECH": 0.011, "BANK": 0.010, "STAPLE": 0.006, "GOLD": 0.009}
alpha = {"MKT_ETF": 0.0, "TECH": 0.00025, "BANK": 0.0, "STAPLE": 0.0001, "GOLD": 0.00015}

rets = pd.DataFrame({k: alpha[k] + betas[k] * factor + rng.normal(0, idio[k], n)
                     for k in betas}, index=dates)
prices = 100 * (1 + rets).cumprod()
port = rets.mean(axis=1).rename("equal_weight")       # our baseline portfolio

print(f"crisis window: {dates[crash_start].date()} to "
      f"{dates[crash_start + crash_len - 1].date()}\n")

summary = pd.DataFrame({
    "ann_return": rets.mean() * 252,
    "ann_vol": rets.std() * np.sqrt(252),
    "max_dd": [((1 + rets[c]).cumprod()
                / (1 + rets[c]).cumprod().cummax() - 1).min() for c in rets],
})
print(summary.round(3))

crisis window: 2019-11-04 to 2020-03-06

         ann_return  ann_vol  max_dd
MKT_ETF       0.078    0.170  -0.584
TECH          0.153    0.290  -0.700
BANK          0.105    0.246  -0.685
STAPLE        0.048    0.134  -0.445
GOLD          0.075    0.148  -0.241


Every asset falls between 24% and 70% peak-to-trough. `GOLD` is the exception because it moves *against* the factor.

The helper below is Module 6's performance report, trimmed to four numbers. We will call it constantly.

In [2]:
def stats(r, label):
    eq = (1 + r).cumprod()
    ann_vol = r.std() * np.sqrt(252)
    return {
        "strategy": label,
        "total": f"{eq.iloc[-1] - 1:.1%}",
        "ann_vol": f"{ann_vol:.1%}",
        "sharpe": round(float((r.mean() * 252) / ann_vol), 2),
        "max_dd": f"{(eq / eq.cummax() - 1).min():.1%}",
    }


print(pd.DataFrame([stats(port, "equal-weight portfolio"),
                    stats(rets["TECH"], "TECH alone"),
                    stats(rets["GOLD"], "GOLD alone")]).to_string(index=False))

              strategy  total ann_vol  sharpe max_dd
equal-weight portfolio  92.4%   14.1%    0.65 -48.4%
            TECH alone 142.6%   29.0%    0.53 -70.0%
            GOLD alone  66.4%   14.8%    0.50 -24.1%


Note what already happened without a single risk rule. The equal-weight portfolio's **Sharpe of 0.65 beats every one of its five holdings** — the best of them on its own is TECH at 0.53 — and its 14.1% volatility is lower than four of the five. It gave up return relative to TECH to get there, but it earned more per unit of risk than anything it contains.

That is diversification, and section 7 measures where it comes from.

## Part 1 — From one file to five components

## 1. Why the monolith stops working

Module 3's structure looks like this:

```python
def on_data(self, data):
    if not self.sma.is_ready: return
    if self.sma.current.value > self.sma_slow.current.value:
        self.set_holdings("SPY", 1.0)
    else:
        self.liquidate()
```

Everything in one place: what to trade, when, how much, and whether to allow it. For one signal on one symbol that is the clearest possible code, and you should keep writing it that way for small algorithms.

Now add requirements. A universe of 50 stocks that refreshes monthly. Two signals whose disagreement needs resolving. Position sizes scaled by volatility. A portfolio drawdown limit. Orders spread out to avoid moving the market.

The `on_data` version of that has all five concerns interleaved, and no way to change one without reading all of it. You cannot answer "what would this do with equal weighting instead?" without a rewrite, and you cannot test the sizing logic without also running the signal.

### The five components

The Framework's answer is to define five interfaces and let data flow one direction through them:

```
Universe Selection  →  which symbols exist
        ↓
Alpha Model         →  Insight objects: "SPY up, for 5 days, confidence 0.6"
        ↓
Portfolio Construction → PortfolioTarget objects: "hold 340 shares of SPY"
        ↓
Risk Management     →  may shrink or cancel those targets
        ↓
Execution           →  places the actual orders
```

| Component | Answers | Emits |
|---|---|---|
| **Universe Selection** | *what can I trade?* | a list of symbols |
| **Alpha** | *what do I predict?* | `Insight` |
| **Portfolio Construction** | *how much of each?* | `PortfolioTarget` |
| **Risk Management** | *is that allowed?* | modified `PortfolioTarget` |
| **Execution** | *how do I get there?* | orders |

The payoff is that each is independently swappable. "Equal weight or volatility weight?" becomes a one-line change with everything else held fixed — which, as Module 6 section 8.3 pointed out, is the only way to tell whether a result depends on a choice you made casually.

> 🧠 **The separation is also a discipline.** The alpha model is forbidden from knowing your account size, and the risk model is forbidden from having an opinion about direction. When those leak into each other you get an algorithm whose behaviour nobody can predict.

## 2. The Alpha model and the `Insight`

An **Insight** is a prediction with an expiry. It says nothing about position size, because that is Portfolio Construction's job.

```python
Insight.price(symbol, timedelta(days=5), InsightDirection.UP)
```

| Field | Meaning |
|---|---|
| `symbol` | what the prediction is about |
| `period` | how long it is valid — after this it expires on its own |
| `direction` | `InsightDirection.UP`, `DOWN` or `FLAT` |
| `magnitude` | optional: expected return, as a decimal |
| `confidence` | optional: 0 to 1 |
| `weight` | optional: requested portfolio fraction |

The expiry is the part worth pausing on. In Module 3's pattern a position persists until some other branch of `on_data` closes it, which is how students end up holding things they cannot explain. An Insight **expires by itself**, and when it does, Portfolio Construction removes the position. Nothing is held by accident.

Here is the crossover signal from Module 5, written as an Alpha model.

In [ ]:
# 🔵 QC cell — a moving-average crossover as an Alpha model
from AlgorithmImports import *
from datetime import timedelta


class MaCrossAlphaModel(AlphaModel):

    def __init__(self, fast=20, slow=60, period_days=5):
        self.fast_period = fast
        self.slow_period = slow
        self.period = timedelta(days=period_days)
        self.state = {}
        self.name = f"MaCross({fast},{slow})"

    def on_securities_changed(self, algorithm, changes):
        # Build indicators when a symbol joins the universe; drop them when it leaves.
        for security in changes.added_securities:
            symbol = security.symbol
            fast = algorithm.sma(symbol, self.fast_period, Resolution.DAILY)
            slow = algorithm.sma(symbol, self.slow_period, Resolution.DAILY)
            self.state[symbol] = {"fast": fast, "slow": slow, "was_above": None}

        for security in changes.removed_securities:
            self.state.pop(security.symbol, None)

    def update(self, algorithm, data):
        insights = []

        for symbol, s in self.state.items():
            if not s["slow"].is_ready:
                continue
            if not data.bars.contains_key(symbol):
                continue

            is_above = s["fast"].current.value > s["slow"].current.value

            # Only emit on a CHANGE of state, not on every bar.
            if s["was_above"] is not None and is_above != s["was_above"]:
                direction = InsightDirection.UP if is_above else InsightDirection.FLAT
                insights.append(Insight.price(symbol, self.period, direction))

            s["was_above"] = is_above

        return insights

Three details that are easy to get wrong.

`on_securities_changed` is where indicators are created, not `initialize`. The universe is not known until the algorithm runs, so this is the only place you learn a symbol exists. Forget the `removed_securities` branch and you leak indicators for every stock that ever entered your universe.

`update` returns `[]` most days. Emitting an insight every bar re-asserts the same view constantly, which fights with Portfolio Construction and generates turnover — and Modules 6 and 8 both showed what turnover does to a live strategy.

The `is_ready` guard is Module 5 section 4 again. Without warm-up, an unready indicator reads `0` and every symbol looks like a downward cross on day one.

## 3. Portfolio Construction: from insight to size

Portfolio Construction receives active (unexpired) insights and returns `PortfolioTarget` objects — *desired end states*, not orders. "Hold 340 shares" rather than "buy 40 shares", which means a missed fill self-corrects on the next pass instead of leaving you with a position you did not intend.

QuantConnect ships several; the built-in you will use most is `EqualWeightingPortfolioConstructionModel`, which splits capital evenly across every symbol with an active non-flat insight.

| Built-in model | Weighting |
|---|---|
| `EqualWeightingPortfolioConstructionModel` | 1/N across active insights |
| `InsightWeightingPortfolioConstructionModel` | proportional to each insight's `weight` |
| `MeanVarianceOptimizationPortfolioConstructionModel` | Markowitz, from historical covariance |
| `RiskParityPortfolioConstructionModel` | equalises each asset's risk contribution |

Before reaching for the sophisticated ones, measure what the simple ones do — which is what Part 2 of this module is for. Mean-variance optimisation estimates a full covariance matrix from noisy history and is famous for producing extreme weights when that estimate is slightly wrong.

Here is a custom one implementing the inverse-volatility rule we will measure in section 9.

In [ ]:
# 🔵 QC cell — inverse-volatility portfolio construction
from AlgorithmImports import *
import numpy as np


class InverseVolatilityPortfolioConstructionModel(PortfolioConstructionModel):

    def __init__(self, lookback=60):
        self.lookback = lookback

    def create_targets(self, algorithm, insights):
        targets = []

        active = [i for i in insights if i.direction != InsightDirection.FLAT]
        if not active:
            # No views: flatten everything the model previously held.
            return [PortfolioTarget(i.symbol, 0) for i in insights]

        vols = {}
        for insight in active:
            history = algorithm.history(insight.symbol, self.lookback, Resolution.DAILY)
            if history.empty:
                continue
            returns = history.loc[insight.symbol]["close"].pct_change().dropna()
            if len(returns) < 2 or returns.std() == 0:
                continue
            vols[insight.symbol] = float(returns.std())

        if not vols:
            return targets

        inverse = {symbol: 1.0 / v for symbol, v in vols.items()}
        total = sum(inverse.values())

        for insight in active:
            if insight.symbol not in inverse:
                continue
            weight = inverse[insight.symbol] / total
            if insight.direction == InsightDirection.DOWN:
                weight = -weight
            targets.append(PortfolioTarget.percent(algorithm, insight.symbol, weight))

        return targets

> ⚠️ **`PortfolioTarget.percent` is a fraction of total portfolio value.** Weights across all targets summing to more than 1.0 means leverage, and the sum of *absolute* weights is what counts — a book that is +0.6 and −0.6 is 1.2 gross, not 0.0. Section 12 covers what that does to you.

## 4. Risk Management and Execution

The **Risk Management** model sees the targets Portfolio Construction produced and may override them. It runs on every data point, not only when insights change, so it can react to a position moving against you between signals.

It returns *only the targets it wants to change*. Returning `[]` means "no objection".

In [ ]:
# 🔵 QC cell — a portfolio-level drawdown limit
from AlgorithmImports import *


class PortfolioDrawdownRiskModel(RiskManagementModel):
    # Flattens the whole book when equity falls `limit` from its high-water mark,
    # and stays flat until it recovers to within `reenter` of that mark.

    def __init__(self, limit=0.15, reenter=0.05):
        self.limit = abs(limit)
        self.reenter = abs(reenter)
        self.peak = 0
        self.halted = False

    def manage_risk(self, algorithm, targets):
        value = algorithm.portfolio.total_portfolio_value
        self.peak = max(self.peak, value)
        if self.peak == 0:
            return []

        drawdown = value / self.peak - 1

        if not self.halted and drawdown <= -self.limit:
            self.halted = True
            algorithm.log(f"RISK HALT at {drawdown:.2%} drawdown")
        elif self.halted and drawdown >= -self.reenter:
            self.halted = False
            algorithm.log(f"RISK RESUME at {drawdown:.2%} drawdown")

        if not self.halted:
            return []

        # Override every target to flat.
        return [PortfolioTarget(symbol, 0)
                for symbol in algorithm.portfolio.keys
                if algorithm.portfolio[symbol].invested]

QuantConnect's built-in risk models cover the common cases, and you should prefer them over hand-rolled versions unless you need something they do not do:

| Model | Effect |
|---|---|
| `MaximumDrawdownPercentPerSecurity(0.05)` | closes any single position down 5% |
| `MaximumDrawdownPercentPortfolio(0.10)` | closes everything at 10% portfolio drawdown |
| `TrailingStopRiskManagementModel(0.05)` | per-position trailing stop |
| `MaximumUnrealizedProfitPercentPerSecurity(0.10)` | takes profit at 10% |
| `MaximumSectorExposureRiskManagementModel(0.20)` | caps exposure to any one sector |

You can add more than one; they are applied in sequence.

### Execution

The **Execution** model turns targets into orders. `ImmediateExecutionModel` sends market orders for the full difference and is the right default for daily-resolution strategies.

| Model | Behaviour |
|---|---|
| `ImmediateExecutionModel` | market orders, immediately |
| `VolumeWeightedAveragePriceExecutionModel` | works the order against volume |
| `StandardDeviationExecutionModel` | waits for a favourable price move |

Module 6 section 1 covered why this matters: a market order crosses the spread, and on an illiquid symbol that cost dwarfs anything your alpha model found.

## 5. Wiring it together

With the components defined, `initialize` becomes a declaration of what the algorithm *is* — five lines, one per component.

In [ ]:
# 🔵 QC cell — a complete framework algorithm
from AlgorithmImports import *
from datetime import timedelta


class FrameworkMomentumAlgorithm(QCAlgorithm):

    def initialize(self):
        self.set_start_date(2016, 1, 1)
        self.set_end_date(2024, 1, 1)
        self.set_cash(100_000)
        self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
                                 AccountType.MARGIN)
        self.set_warm_up(90, Resolution.DAILY)

        symbols = [Symbol.create(t, SecurityType.EQUITY, Market.USA)
                   for t in ["SPY", "QQQ", "XLF", "XLP", "GLD"]]

        # 1. what can I trade?
        self.set_universe_selection(ManualUniverseSelectionModel(symbols))
        # 2. what do I predict?
        self.set_alpha(MaCrossAlphaModel(fast=20, slow=60, period_days=5))
        # 3. how much of each?
        self.set_portfolio_construction(InverseVolatilityPortfolioConstructionModel(60))
        # 4. is that allowed?
        self.add_risk_management(PortfolioDrawdownRiskModel(0.15, 0.05))
        self.add_risk_management(MaximumDrawdownPercentPerSecurity(0.10))
        # 5. how do I get there?
        self.set_execution(ImmediateExecutionModel())

        self.settings.rebalance_portfolio_on_insight_changes = True
        self.settings.free_portfolio_value_percentage = 0.05   # cash buffer

Read that `initialize` and you know what the algorithm does without opening another file. That is the whole argument for the Framework.

Two settings worth knowing. `rebalance_portfolio_on_insight_changes` controls whether new insights trigger an immediate rebalance or wait for a schedule — leaving it on with a chatty alpha model produces constant turnover. `free_portfolio_value_percentage` holds back cash so that a target requiring slightly more capital than you have does not trigger a margin call on a price gap.

> ⚠️ **A framework algorithm is not automatically better than a monolithic one.** It is more testable and more swappable. If your strategy is one signal on one symbol, Module 3's pattern is clearer and you should use it. Reach for the Framework when you have components worth swapping.

### ✏️ Your turn — read the wiring

No code for this one — answer from the algorithm above by filling in a dict called `wiring` with these exact keys and string values:

| key | question |
|---|---|
| `"emits_insights"` | which component produces `Insight` objects? |
| `"decides_size"` | which component decides position size? |
| `"can_veto"` | which component can override a target? |
| `"places_orders"` | which component sends orders to the broker? |

Use exactly one of these strings for each: `"universe"`, `"alpha"`, `"portfolio_construction"`, `"risk"`, `"execution"`.

In [3]:
wiring = {
    "emits_insights": "alpha",
    "decides_size": "portfolio_construction",
    "can_veto": "risk",
    "places_orders": "execution",
}

for k, v in wiring.items():
    print(f"{k:18s} -> {v}")

emits_insights     -> alpha
decides_size       -> portfolio_construction
can_veto           -> risk
places_orders      -> execution


In [4]:
EXPECTED = {
    "emits_insights": "alpha",
    "decides_size": "portfolio_construction",
    "can_veto": "risk",
    "places_orders": "execution",
}
assert set(wiring) == set(EXPECTED), f"keys must be exactly {sorted(EXPECTED)}"
for k, v in EXPECTED.items():
    assert wiring[k] == v, (
        f"'{k}' should be '{v}', got '{wiring[k]}'. Re-read the data-flow diagram "
        "in section 1: insights flow alpha -> portfolio construction -> risk -> execution.")
print("✅ Correct!  Alpha predicts, portfolio construction sizes,",
      "risk vetoes, execution trades. Each stage may only do its own job.")

✅ Correct!  Alpha predicts, portfolio construction sizes, risk vetoes, execution trades. Each stage may only do its own job.


## Part 2 — Risk management, measured

## 6. Why drawdown is the number that matters

Module 6 section 4 introduced drawdown. Here is the arithmetic that makes it the constraint everything else bends around.

A loss and the gain that undoes it are not symmetric, because the gain is computed on a smaller base. Lose 50% and you need **100%** to get back, not 50%.

In [5]:
dd = np.array([0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90])

recovery = pd.DataFrame({
    "drawdown": [f"-{d:.0%}" for d in dd],
    "gain_to_recover": [f"+{1 / (1 - d) - 1:.0%}" for d in dd],
    "years_at_10%_a_year": [round(np.log(1 / (1 - d)) / np.log(1.10), 1) for d in dd],
})
print(recovery.to_string(index=False))

drawdown gain_to_recover  years_at_10%_a_year
     -5%             +5%                  0.5
    -10%            +11%                  1.1
    -20%            +25%                  2.3
    -30%            +43%                  3.7
    -50%           +100%                  7.3
    -70%           +233%                 12.6
    -90%           +900%                 24.2


The right-hand column is the one to sit with. Our equal-weight portfolio's **−48% drawdown** takes roughly **7 years** of 10%-a-year returns to undo.

And that is the arithmetic alone. In practice a drawdown that size ends the strategy before the recovery arrives, because someone — a risk committee, a competition rule, or you at 2am — turns it off first. Module 6 made this point in one line and it is worth the repetition: **the maximum drawdown you can tolerate is a hard constraint on strategy selection, not a statistic you report afterwards.**

> 🧠 **This is why risk management is worth paying for.** Sections 8 to 11 all cost something. What they buy is staying in the game, which is a prerequisite for the edge you spent eight modules finding.

### ✏️ Your turn — the recovery table

Write `recovery_gain(drawdown)` returning the fractional gain needed to recover from a fractional `drawdown` — so `recovery_gain(0.5)` is `1.0` (a 100% gain).

Then write `survivable(drawdown, annual_return, max_years)` returning `True` if the recovery can be achieved within `max_years` at a compound `annual_return`.

Set `gain_50` to `recovery_gain(0.5)`, and `ok_20` / `ok_60` to `survivable` at drawdowns of 0.20 and 0.60 with `annual_return=0.10, max_years=3`.

In [6]:
def recovery_gain(drawdown):
    return 1.0 / (1.0 - drawdown) - 1.0


def survivable(drawdown, annual_return, max_years):
    needed = recovery_gain(drawdown)
    achievable = (1.0 + annual_return) ** max_years - 1.0
    return bool(achievable >= needed)


gain_50 = recovery_gain(0.5)
ok_20 = survivable(0.20, 0.10, 3)
ok_60 = survivable(0.60, 0.10, 3)

print(f"50% drawdown needs a {gain_50:.0%} gain")
print(f"recover 20% within 3 years at 10%/yr: {ok_20}")
print(f"recover 60% within 3 years at 10%/yr: {ok_60}")

50% drawdown needs a 100% gain
recover 20% within 3 years at 10%/yr: True
recover 60% within 3 years at 10%/yr: False


In [7]:
assert np.isclose(gain_50, 1.0), f"a 50% drawdown needs a 100% gain, got {gain_50:.4f}"
assert np.isclose(recovery_gain(0.0), 0.0), "no drawdown needs no gain"
assert np.isclose(recovery_gain(0.2), 0.25), \
    f"a 20% drawdown needs a 25% gain, got {recovery_gain(0.2):.4f}"
assert np.isclose(recovery_gain(0.9), 9.0), \
    f"a 90% drawdown needs a 900% gain, got {recovery_gain(0.9):.4f}"
assert ok_20 is True, "10%/yr for 3 years is +33%, which clears the 25% needed after a 20% fall"
assert ok_60 is False, "a 60% drawdown needs +150%; 3 years at 10% gives only +33%"
# The threshold must be genuine, not hard-coded.
assert survivable(0.20, 0.50, 1) is True, "a 50% year clears a 20% drawdown"
assert survivable(0.20, 0.01, 1) is False, "a 1% year does not clear a 20% drawdown"
print(f"✅ Correct!  A 50% loss needs a {gain_50:.0%} gain to undo.",
      "Drawdown is not a statistic you report - it is a constraint you design around.")

✅ Correct!  A 50% loss needs a 100% gain to undo. Drawdown is not a statistic you report - it is a constraint you design around.


## 7. Diversification: the one technique that is free

Every other method in this module trades return for safety. Diversification does not, which is why it comes first.

Module 1 section 10 gave the machinery: portfolio variance is `w.T @ cov @ w`, and it is **less** than the weighted average of individual variances whenever assets are less than perfectly correlated. Let us see how much less.

In [8]:
w = np.repeat(1 / 5, 5)
cov_annual = rets.cov() * 252

portfolio_vol = float(np.sqrt(w @ cov_annual @ w))
weighted_avg_vol = float(w @ (rets.std() * np.sqrt(252)))

print(f"weighted average of the five vols : {weighted_avg_vol:.2%}")
print(f"actual portfolio volatility       : {portfolio_vol:.2%}")
print(f"reduction from diversification    : {1 - portfolio_vol / weighted_avg_vol:.1%}")
print()
print("correlation matrix:")
print(rets.corr().round(2))

weighted average of the five vols : 19.76%
actual portfolio volatility       : 14.05%
reduction from diversification    : 28.9%

correlation matrix:
         MKT_ETF  TECH  BANK  STAPLE  GOLD
MKT_ETF     1.00  0.78  0.76    0.69 -0.27
TECH        0.78  1.00  0.61    0.56 -0.21
BANK        0.76  0.61  1.00    0.53 -0.23
STAPLE      0.69  0.56  0.53    1.00 -0.20
GOLD       -0.27 -0.21 -0.23   -0.20  1.00


**29% of the volatility disappeared** and no return was given up to get it. That is the closest thing to a free lunch in finance, and it comes entirely from the correlation matrix — `GOLD` at −0.2 against everything else is doing most of the work.

To prove that, remove one asset at a time and re-measure.

In [9]:
rows = []
for drop in rets.columns:
    kept = rets.drop(columns=drop)
    wk = np.repeat(1 / len(kept.columns), len(kept.columns))
    rows.append({
        "removed": drop,
        "own_vol": f"{rets[drop].std() * np.sqrt(252):.1%}",
        "avg_corr_to_rest": round(float(rets.corr()[drop].drop(drop).mean()), 2),
        "portfolio_vol_without_it": f"{float(np.sqrt(wk @ (kept.cov() * 252) @ wk)):.2%}",
    })

print(f"portfolio vol with all five: {portfolio_vol:.2%}\n")
print(pd.DataFrame(rows).to_string(index=False))

portfolio vol with all five: 14.05%

removed own_vol  avg_corr_to_rest portfolio_vol_without_it
MKT_ETF   17.0%              0.49                   13.86%
   TECH   29.0%              0.44                   11.72%
   BANK   24.6%              0.41                   12.87%
 STAPLE   13.4%              0.39                   15.29%
   GOLD   14.8%             -0.23                   18.17%


Read that table carefully, because it contradicts the obvious intuition.

Removing **TECH** — by far the most volatile asset at 29% — *lowers* portfolio volatility to 11.7%. Removing **GOLD**, which is only mid-volatility at 15%, *raises* it to 18.2%, the worst outcome of the five.

The `avg_corr_to_rest` column explains it. What an asset contributes to portfolio risk is its **covariance** with everything else, and covariance is correlation *times* the two volatilities. Both terms matter, and they can pull in opposite directions.

- **TECH** has a middling correlation (0.44) but an enormous volatility (29.0%). The volatility term dominates, so it adds risk on net and removing it helps.
- **GOLD** has an unremarkable volatility (14.8%) but a *negative* correlation (−0.23). The sign flips its contribution, so it subtracts risk and removing it hurts more than removing anything else.

> 🧠 **You cannot rank holdings by risk contribution using volatility alone.** A volatile asset that moves against the others can lower total portfolio risk; a calm asset that moves with everything else cannot. This is why "I diversified, I own 20 tech stocks" is not diversification — twenty correlated positions behave like one large one.

The practical version: when you add a symbol to your universe, the question is not *is it risky?* but *does it move with what I already own?*

### ✏️ Your turn — measure the diversification benefit

Write `diversification_ratio(returns, weights=None)` returning the ratio

> (weighted average of individual annualized vols) ÷ (annualized portfolio vol)

as a float. A ratio of 1.0 means no benefit; higher is better. Default to equal weights when `weights` is `None`. Annualize with `√252`.

Then write `worst_to_remove(returns)` returning the **column name** whose removal raises the equal-weighted portfolio volatility the most — that is, the asset doing the most diversifying.

Set `dr_all` to the ratio for all five assets, `dr_no_gold` to the ratio without `GOLD`, and `most_valuable` to `worst_to_remove(rets)`.

In [10]:
def diversification_ratio(returns, weights=None):
    if weights is None:
        weights = np.repeat(1 / returns.shape[1], returns.shape[1])
    weights = np.asarray(weights, dtype=float)

    individual = returns.std().values * np.sqrt(252)
    weighted_avg = float(weights @ individual)

    portfolio = float(np.sqrt(weights @ (returns.cov().values * 252) @ weights))
    return weighted_avg / portfolio


def worst_to_remove(returns):
    best_name, best_vol = None, -np.inf

    for col in returns.columns:
        kept = returns.drop(columns=col)
        wk = np.repeat(1 / kept.shape[1], kept.shape[1])
        vol = float(np.sqrt(wk @ (kept.cov().values * 252) @ wk))
        if vol > best_vol:
            best_name, best_vol = col, vol

    return best_name


dr_all = diversification_ratio(rets)
dr_no_gold = diversification_ratio(rets.drop(columns="GOLD"))
most_valuable = worst_to_remove(rets)

print(f"ratio with all five : {dr_all:.3f}")
print(f"ratio without GOLD  : {dr_no_gold:.3f}")
print(f"best diversifier    : {most_valuable}")

ratio with all five : 1.406
ratio without GOLD  : 1.156
best diversifier    : GOLD


In [11]:
_w = np.repeat(1 / 5, 5)
_exp = float(_w @ (rets.std().values * np.sqrt(252))) / float(np.sqrt(_w @ (rets.cov().values * 252) @ _w))
assert np.isclose(dr_all, _exp), f"dr_all should be {_exp:.4f}, got {dr_all:.4f}"
assert dr_all > 1.3, "with a negative-correlation asset the ratio should be well above 1"
assert dr_no_gold < dr_all, "removing the diversifier must lower the ratio"
assert most_valuable == "GOLD", (
    f"got '{most_valuable}'. The best diversifier is the one whose removal HURTS most "
    "(raises portfolio vol), not the one with the lowest volatility.")
# A single asset cannot diversify itself.
assert np.isclose(diversification_ratio(rets[["TECH"]]), 1.0), \
    "a one-asset portfolio must have a ratio of exactly 1.0"
# Perfectly correlated assets give no benefit.
_dup = pd.DataFrame({"a": rets["TECH"], "b": rets["TECH"]})
assert np.isclose(diversification_ratio(_dup), 1.0), \
    "two copies of the same asset must give a ratio of 1.0 - no diversification"
print(f"✅ Correct!  Ratio {dr_all:.3f} with all five, {dr_no_gold:.3f} without GOLD.",
      f"'{most_valuable}' is the most valuable holding despite being neither the",
      "highest-returning nor the lowest-volatility asset.")

✅ Correct!  Ratio 1.406 with all five, 1.156 without GOLD. 'GOLD' is the most valuable holding despite being neither the highest-returning nor the lowest-volatility asset.


## 8. Position sizing: inverse volatility

Equal weighting gives every asset the same *capital*, which gives the volatile ones far more *risk*. TECH at 29% vol and STAPLE at 13% vol are not equal partners in a 1/N portfolio; TECH drives it.

**Inverse-volatility weighting** equalises risk instead of capital: weight each asset by `1/σ`, normalised. Everything is estimated on a trailing window and `shift(1)`-ed, which is Module 5 section 8's rule — the weight for tomorrow may only use volatility known today.

In [12]:
LOOKBACK = 60

rolling_vol = rets.rolling(LOOKBACK).std()
inv = 1.0 / rolling_vol
weights = inv.div(inv.sum(axis=1), axis=0).shift(1)      # shift = no look-ahead

inv_vol_port = (weights * rets).sum(axis=1)
start = rets.index[LOOKBACK]                              # common start for fair comparison

print("average weight per asset:")
print(weights.mean().round(3).to_string())
print()
print(pd.DataFrame([stats(port.loc[start:], "equal weight"),
                    stats(inv_vol_port.loc[start:], "inverse volatility")]
                   ).to_string(index=False))

average weight per asset:
MKT_ETF    0.225
TECH       0.125
BANK       0.147
STAPLE     0.268
GOLD       0.235

          strategy total ann_vol  sharpe max_dd
      equal weight 82.6%   14.1%    0.62 -48.4%
inverse volatility 88.4%   11.8%    0.75 -36.9%


STAPLE gets 26.8% of the book and TECH gets 12.5% — almost exactly the inverse of their volatilities.

The result improves **all four** statistics: higher return, lower volatility, higher Sharpe, smaller drawdown. That combination is rare, and the reason it happens here is that TECH's extra volatility was not buying proportionally extra return. Inverse-vol weighting is not magic; it is a bet that risk-adjusted returns are more similar across assets than raw returns are, which is usually closer to true than the alternative.

> ⚠️ **Inverse-vol ignores correlation entirely.** It would happily put 60% of the book into three highly correlated low-vol assets. Risk parity is the version that accounts for correlation, and `RiskParityPortfolioConstructionModel` implements it on QuantConnect.

### ✏️ Your turn — inverse-volatility weights

Write `inverse_vol_weights(returns, lookback=60)` returning a DataFrame of weights, same shape and index as `returns`, where each row is `1/σ` normalised to sum to 1 and **shifted forward one day** so no weight uses same-day information. Rows before the lookback completes will be `NaN`.

Then write `apply_weights(returns, weights)` returning the portfolio return Series (`(weights × returns).sum(axis=1)`), with rows where the weights are `NaN` dropped.

Build `w30 = inverse_vol_weights(rets, 30)` and `p30 = apply_weights(rets, w30)`, then set `sharpe30` to its annualized Sharpe.

In [13]:
def inverse_vol_weights(returns, lookback=60):
    vol = returns.rolling(lookback).std()
    inv = 1.0 / vol
    return inv.div(inv.sum(axis=1), axis=0).shift(1)


def apply_weights(returns, weights):
    combined = (weights * returns).sum(axis=1)
    return combined[weights.notna().all(axis=1)]


w30 = inverse_vol_weights(rets, 30)
p30 = apply_weights(rets, w30)
sharpe30 = float((p30.mean() * 252) / (p30.std() * np.sqrt(252)))

print(f"first valid weight row: {w30.dropna().index[0].date()}")
print(w30.dropna().iloc[0].round(3).to_string())
print(f"\n30-day inverse-vol Sharpe: {sharpe30:.3f}")

first valid weight row: 2016-02-15
MKT_ETF    0.218
TECH       0.134
BANK       0.139
STAPLE     0.272
GOLD       0.236

30-day inverse-vol Sharpe: 0.829


In [14]:
assert w30.shape == rets.shape, f"weights must match the shape of returns, got {w30.shape}"
_valid = w30.dropna()
assert np.allclose(_valid.sum(axis=1), 1.0), "every weight row must sum to 1"
assert (_valid >= 0).all().all(), "inverse-vol weights are all positive"
# The shift is what prevents look-ahead: row t must use vol computed to t-1.
_expect = (1.0 / rets.rolling(30).std())
_expect = _expect.div(_expect.sum(axis=1), axis=0).shift(1)
assert np.allclose(_valid.values, _expect.loc[_valid.index].values), \
    "weights are off - normalise 1/vol across each row, THEN .shift(1)"
assert w30.iloc[:30].isna().all().all(), \
    "rows before the lookback completes must be NaN, and the shift adds one more"
# Low-vol assets must get more weight than high-vol ones.
_last = _valid.iloc[-1]
assert _last["STAPLE"] > _last["TECH"], \
    "the lower-volatility asset must receive the larger weight"
assert len(p30) == len(_valid), "apply_weights should return one row per valid weight row"
assert np.isclose(sharpe30, float((p30.mean() * 252) / (p30.std() * np.sqrt(252)))), \
    "sharpe30 should be the annualized Sharpe of p30"
print(f"✅ Correct!  A 30-day lookback gives Sharpe {sharpe30:.3f}.",
      "Weights are normalised, shifted, and inversely ordered by volatility.")

✅ Correct!  A 30-day lookback gives Sharpe 0.829. Weights are normalised, shifted, and inversely ordered by volatility.


## 9. Volatility targeting

Inverse-vol decides *relative* sizes. Volatility targeting decides the *total*: scale the whole book so that its expected volatility hits a chosen number.

```
scale = target_vol / recent_realised_vol
```

Volatility clusters — calm follows calm, turbulence follows turbulence — so recent realised volatility is a genuinely useful forecast of near-term volatility, in a way that recent *returns* are not a useful forecast of near-term returns. That asymmetry is what makes this work.

We cap leverage at 2× so a quiet stretch cannot produce an absurd position.

In [15]:
TARGET_VOL = 0.12

realised = port.rolling(60).std() * np.sqrt(252)
scale = (TARGET_VOL / realised).shift(1).clip(upper=2.0)     # shift: yesterday's estimate
vol_targeted = scale * port

print(pd.DataFrame([stats(port.loc[start:], "equal weight (unscaled)"),
                    stats(vol_targeted.loc[start:], f"vol-targeted {TARGET_VOL:.0%}")]
                   ).to_string(index=False))
print()
print(f"leverage: average {scale.loc[start:].mean():.2f}, "
      f"range {scale.loc[start:].min():.2f} to {scale.loc[start:].max():.2f}")

               strategy  total ann_vol  sharpe max_dd
equal weight (unscaled)  82.6%   14.1%    0.62 -48.4%
       vol-targeted 12% 137.6%   12.3%    0.96 -29.4%

leverage: average 0.95, range 0.35 to 1.26


Realised volatility came out at 12.3% against a 12% target, so the mechanism does what it claims. Return rose, drawdown fell, and Sharpe went from 0.62 to 0.96.

**Be precise about why it worked**, because the reason is not "less risk is better". Watch the leverage move through the crisis.

In [16]:
checkpoints = {
    "60 days before": crash_start - 60,
    "day before": crash_start - 1,
    "20 days in": crash_start + 20,
    "60 days in": crash_start + 60,
    "200 days after": crash_start + 200,
}

for label, i in checkpoints.items():
    print(f"{label:>16s} : leverage {float(scale.iloc[i]):.2f}   "
          f"trailing vol {float(realised.iloc[i]):.1%}")

  60 days before : leverage 0.92   trailing vol 13.2%
      day before : leverage 1.07   trailing vol 11.6%
      20 days in : leverage 0.52   trailing vol 23.4%
      60 days in : leverage 0.37   trailing vol 31.8%
  200 days after : leverage 1.05   trailing vol 11.2%


Look at the first two rows before the crisis. Sixty days out the scale was 0.92; the day before it was **1.07**. Trailing volatility had *fallen* to 11.6% during the calm run-up, so the rule levered **up** into the crash. It then cut to 0.52 twenty days in and 0.37 by day sixty.

It predicted nothing. It reacted, with a lag, after volatility had already risen — and by then part of the damage was done.

What saved return was that the crisis was *long*. Volatility stayed elevated for months while the drift stayed negative, so a de-risked book missed most of the decline. Had the crash been a single-day gap, this would have provided no protection at all.

> ⚠️ **Volatility targeting helps when high volatility coincides with poor returns.** That is the historical norm in equities, not a law. In a violent upward move it scales you *out* of the best days. And it is a lagging indicator by construction: it can never protect you from the first move, only from the fifth.

One more cost that this simulation does not charge you for: scaling the book means trading it. Rebalancing exposure daily generates turnover, and Module 6 section 6 showed what turnover costs. In practice you rebalance on a band — only when the scale drifts more than, say, 20% from its current value.

### ✏️ Your turn — volatility targeting

Write `vol_target_scale(returns, target=0.12, lookback=60, max_leverage=2.0)` returning the leverage Series: `target ÷ trailing annualized vol`, **shifted one day**, and clipped above at `max_leverage`.

Then write `realised_vol(returns)` returning the annualized volatility of a return Series as a float.

Apply it at a 20% target to build `vt20`, and set `vt20_vol` to its realised volatility. It should land near 20%, well above the 12% version.

In [17]:
def vol_target_scale(returns, target=0.12, lookback=60, max_leverage=2.0):
    trailing = returns.rolling(lookback).std() * np.sqrt(252)
    return (target / trailing).shift(1).clip(upper=max_leverage)


def realised_vol(returns):
    return float(returns.std() * np.sqrt(252))


scale20 = vol_target_scale(port, target=0.20)
vt20 = (scale20 * port).dropna()
vt20_vol = realised_vol(vt20)

print(f"realised vol at a 20% target : {vt20_vol:.2%}")
print(f"average leverage             : {scale20.mean():.2f}")

realised vol at a 20% target : 20.57%
average leverage             : 1.58


In [18]:
_trail = port.rolling(60).std() * np.sqrt(252)
_exp = (0.20 / _trail).shift(1).clip(upper=2.0)
assert np.allclose(scale20.dropna().values, _exp.dropna().values), \
    "scale is off - annualize the trailing vol, divide target by it, THEN shift and clip"
assert scale20.max() <= 2.0 + 1e-12, "leverage must be capped at max_leverage"
assert scale20.dropna().index[0] > port.index[59], "the scale must be shifted, not same-day"
assert np.isclose(realised_vol(port), float(port.std() * np.sqrt(252))), "realised_vol is off"
# A higher target must produce more leverage and more volatility.
_s12 = vol_target_scale(port, target=0.12)
assert scale20.mean() > _s12.mean(), "a 20% target should use more leverage than a 12% target"
assert 0.15 < vt20_vol < 0.26, \
    f"a 20% target should land near 20% realised, got {vt20_vol:.2%}"
assert vt20_vol > realised_vol((_s12 * port).dropna()), \
    "the 20% target must realise higher volatility than the 12% target"
print(f"✅ Correct!  A 20% target realised {vt20_vol:.2%} at {scale20.mean():.2f}x average",
      "leverage. The estimate is backward-looking, so it lags every turn in the market.")

✅ Correct!  A 20% target realised 20.57% at 1.58x average leverage. The estimate is backward-looking, so it lags every turn in the market.


## 10. The drawdown kill switch, and what it really costs

Here is the rule most students propose first: if the portfolio falls X% from its high-water mark, go to cash; come back when it recovers.

It works. It is also expensive, and the measurement below is the honest version that most write-ups skip.

The implementation detail that decides whether this is a real rule or a broken one: while you are flat, you must keep tracking what the market *would* have done. Otherwise your equity is frozen, the drawdown never recovers, and you never re-enter.

In [19]:
def drawdown_switch(returns, limit=0.15, reenter=0.05):
    shadow, peak, halted = 1.0, 1.0, False
    out = []

    for r in returns:
        out.append(0.0 if halted else r)

        shadow *= (1 + r)                # what the market did, whether or not we were in
        peak = max(peak, shadow)
        drawdown = shadow / peak - 1

        if not halted and drawdown <= -limit:
            halted = True
        elif halted and drawdown >= -reenter:
            halted = False

    return pd.Series(out, index=returns.index)


rows = [stats(port, "no risk control")]
for limit in [0.10, 0.15, 0.20, 0.25]:
    switched = drawdown_switch(port, limit)
    row = stats(switched, f"flat below -{limit:.0%}")
    row["days_flat"] = int((switched == 0).sum())
    rows.append(row)

print(pd.DataFrame(rows).fillna("").to_string(index=False))

       strategy total ann_vol  sharpe max_dd days_flat
no risk control 92.4%   14.1%    0.65 -48.4%          
flat below -10% 57.6%    9.7%    0.63 -15.9%     896.0
flat below -15% 63.0%   10.5%    0.63 -22.0%     710.0
flat below -20% 58.6%   11.0%    0.58 -28.2%     613.0
flat below -25% 49.0%   11.1%    0.50 -32.6%     607.0


Every single setting **loses money relative to doing nothing** — 92.4% becomes 49% to 63% — and not one of them improves the Sharpe ratio. What they buy is the drawdown column: −48.4% becomes −15.9% to −32.6%.

That is the entire trade, stated plainly:

> 🧠 **A drawdown kill switch is insurance, not alpha.** You pay a premium in return and you receive a smaller worst case. It does not improve risk-adjusted performance, and any backtest that says otherwise has probably fitted the threshold to one crisis.

Whether the premium is worth paying depends on something outside the backtest. If a 48% drawdown means you get shut down — by a competition rule, a risk limit, or your own nerve — then a rule that costs 30 points of return and holds the loss to 22% is obviously correct, because the alternative was never "hold on and recover", it was "stop trading at the bottom".

Now compare each limit against the drawdown it actually delivered:

| limit set | drawdown realised | ratio | days flat |
|---|---|---|---|
| −10% | −15.9% | 1.59× | 896 |
| −15% | −22.0% | 1.47× | 710 |
| −20% | −28.2% | 1.41× | 613 |
| −25% | −32.6% | 1.30× | 607 |

**Not one setting delivered the limit it was given.** Every realised drawdown overshot by 30% to 60%.

Two things cause that, and neither is fixable by choosing a better number. The switch triggers *after* the loss has already happened, so the threshold is a detection level rather than a floor. And re-entry at −5% puts you back in during a decline that has not finished, so a long fall is absorbed in several bites that compound.

> ⚠️ **Set the limit well inside the drawdown you can actually tolerate.** If your hard ceiling is 20%, a 20% kill switch will breach it — it delivered −28.2% here. On this data a 14% limit was the loosest that stayed under 20%, and even that number is fitted to one crisis and would not transfer.

Also note that return does not order itself by limit at all: 57.6%, 63.0%, 58.6%, 49.0% going from tight to loose. There is no "correct" threshold to be found here, only the noise you would be fitting if you went looking for one.

### ✏️ Your turn — the drawdown switch

Write `risk_switch(returns, limit=0.15, reenter=0.05)` returning a **position Series** (`1.0` when invested, `0.0` when halted) rather than the returns themselves.

Track a shadow equity that compounds `returns` on **every** day regardless of position, and its running peak. Halt when the shadow drawdown is `<= -limit`; resume when it recovers to `>= -reenter`. The position for a given day reflects the state **before** that day's return is applied.

Then set `pos15` to the positions at a 15% limit, `switched15` to `pos15 * port`, and `dd_reduction` to `(baseline max drawdown) − (switched max drawdown)` as a positive float of absolute drawdown reduced.

In [20]:
def risk_switch(returns, limit=0.15, reenter=0.05):
    shadow, peak, halted = 1.0, 1.0, False
    positions = []

    for r in returns:
        positions.append(0.0 if halted else 1.0)

        shadow *= (1 + r)
        peak = max(peak, shadow)
        drawdown = shadow / peak - 1

        if not halted and drawdown <= -limit:
            halted = True
        elif halted and drawdown >= -reenter:
            halted = False

    return pd.Series(positions, index=returns.index)


def max_drawdown(r):
    eq = (1 + r).cumprod()
    return float((eq / eq.cummax() - 1).min())


pos15 = risk_switch(port, 0.15)
switched15 = pos15 * port
dd_reduction = abs(max_drawdown(port)) - abs(max_drawdown(switched15))

print(f"days flat        : {int((pos15 == 0).sum())}")
print(f"baseline max dd  : {max_drawdown(port):.1%}")
print(f"switched max dd  : {max_drawdown(switched15):.1%}")
print(f"reduction        : {dd_reduction:.1%}")

days flat        : 710
baseline max dd  : -48.4%
switched max dd  : -22.0%
reduction        : 26.4%


In [21]:
assert set(pos15.unique()) <= {0.0, 1.0}, "positions must be exactly 0.0 or 1.0"
assert pos15.iloc[0] == 1.0, "you start invested, before any drawdown has occurred"
assert 0 < (pos15 == 0).sum() < len(pos15), "the switch must both fire and recover"
# The shadow equity must keep tracking the market while flat, or it can never re-enter.
assert int(((pos15 == 0).astype(int).diff() != 0).sum()) > 2, (
    "the switch should halt AND resume more than once. If it never resumes, your shadow "
    "equity is frozen while flat - it must compound every day regardless of position.")
assert dd_reduction > 0.15, \
    f"a 15% switch should cut the drawdown substantially, got {dd_reduction:.3f}"
assert abs(max_drawdown(switched15)) < abs(max_drawdown(port)), \
    "the switched series must have a smaller drawdown"
# And the honest part: it costs return.
assert (1 + switched15).cumprod().iloc[-1] < (1 + port).cumprod().iloc[-1], \
    "the switch should REDUCE total return - if yours increases it, re-check the timing"
print(f"✅ Correct!  The switch cut the drawdown by {dd_reduction:.1%}",
      f"and spent {int((pos15 == 0).sum())} days in cash to do it.",
      "That trade is the whole point - it is insurance, and insurance has a premium.")

✅ Correct!  The switch cut the drawdown by 26.4% and spent 710 days in cash to do it. That trade is the whole point - it is insurance, and insurance has a premium.


## 11. Trailing stops, and why they disappoint

The trailing stop is the technique every beginner reaches for and the one with the weakest evidence behind it. Exit when price falls X% from its high since entry; re-enter after a bounce.

Applied to TECH, our most volatile asset, across three widths.

In [22]:
def trailing_stop(px, pct=0.20, rebound=0.10):
    position, high, low = 1, px.iloc[0], px.iloc[0]
    out = []

    for p in px:
        if position == 1:
            high = max(high, p)
            if p <= high * (1 - pct):
                position, low = 0, p
        else:
            low = min(low, p)
            if p >= low * (1 + rebound):
                position, high = 1, p
        out.append(position)

    return pd.Series(out, index=px.index)


rows = [stats(rets["TECH"], "buy & hold")]
for pct in [0.15, 0.20, 0.30]:
    pos = trailing_stop(prices["TECH"], pct).shift(1).fillna(1)
    row = stats((pos * rets["TECH"]).dropna(), f"{pct:.0%} trailing stop")
    row["exits"] = int((pos.diff() == -1).sum())
    row["days_out"] = int((pos == 0).sum())
    rows.append(row)

print(pd.DataFrame(rows).fillna("").to_string(index=False))

         strategy  total ann_vol  sharpe max_dd exits days_out
       buy & hold 142.6%   29.0%    0.53 -70.0%               
15% trailing stop 109.0%   25.3%    0.49 -55.5%  13.0    311.0
20% trailing stop 169.3%   26.5%    0.60 -55.5%   7.0    135.0
30% trailing stop 209.6%   27.4%    0.65 -55.7%   3.0     70.0


Three things in that table, in order of importance.

**1. The drawdown barely moved.** Buy and hold loses 70.0%; all three of these settings still lose about 55.5%. The stop cannot protect you from the crisis because it only fires *after* price has already fallen X% — and then the market keeps falling while you wait for the rebound signal to bring you back. That is section 10's overshoot problem again, at the single-position level.

**2. The returns swing wildly with the parameter.** 109%, 169%, 210% across three settings, with no monotonic pattern in drawdown to justify it. Module 6 section 8.3 called this parameter instability, and it is the signature of a rule whose backtest performance is luck.

**3. The whole difference rests on a handful of decisions.** The 30% stop exits **three times in eight years**. Its outperformance is essentially one well-timed exit. Module 6 section 8.5 was explicit: with three observations you have no evidence at all, and you certainly cannot tell whether 30% is better than 20%.

> ⚠️ **Do not tune a stop-loss width on a backtest.** You will find a value that dodged the worst episode in your sample, and you will have learned nothing about the next one. If you use stops, pick the width from position-sizing logic (how much you can afford to lose per position) and leave it alone.

Compare this against section 7's diversification result, which came from a structural property of the correlation matrix and did not depend on tuning anything. That contrast is the point of Part 2: the techniques that survive out of sample are the ones that do not need a parameter fitted to a crisis you have already seen.

### ✏️ Your turn — stop-width stability

You are testing whether a rule's performance depends on a parameter you cannot know in advance.

Write `stop_sweep(px, r, widths)` returning a DataFrame indexed by `width` with columns `total_return`, `max_dd` and `exits` (all floats/ints, **not** formatted strings) for each trailing-stop width. Use the `trailing_stop` helper above, `.shift(1).fillna(1)` the positions, and `.dropna()` the returns.

Then write `is_stable(sweep, tolerance=0.25)` returning `True` only if the spread of `total_return` across widths — `(max − min)` — is **less than** `tolerance` in absolute return terms.

Run the sweep over `[0.10, 0.15, 0.20, 0.25, 0.30]` as `sweep`, and set `stable` to `is_stable(sweep)`. Expect `False` — and read the `max_dd` column too, not just the returns.

In [23]:
def stop_sweep(px, r, widths):
    rows = []
    for w in widths:
        pos = trailing_stop(px, w).shift(1).fillna(1)
        net = (pos * r).dropna()
        eq = (1 + net).cumprod()
        rows.append({
            "width": w,
            "total_return": float(eq.iloc[-1] - 1),
            "max_dd": float((eq / eq.cummax() - 1).min()),
            "exits": int((pos.diff() == -1).sum()),
        })

    return pd.DataFrame(rows).set_index("width")


def is_stable(sweep, tolerance=0.25):
    spread = sweep["total_return"].max() - sweep["total_return"].min()
    return bool(spread < tolerance)


sweep = stop_sweep(prices["TECH"], rets["TECH"], [0.10, 0.15, 0.20, 0.25, 0.30])
stable = is_stable(sweep)

print(sweep.round(3).to_string())
print(f"\nspread {sweep['total_return'].max() - sweep['total_return'].min():.1%}"
      f"  -> stable: {stable}")

       total_return  max_dd  exits
width                             
0.10          2.267  -0.363     19
0.15          1.090  -0.555     13
0.20          1.693  -0.555      7
0.25          2.508  -0.519      3
0.30          2.096  -0.557      3

spread 141.9%  -> stable: False


In [24]:
assert list(sweep.columns) == ["total_return", "max_dd", "exits"], \
    f"columns must be exactly ['total_return', 'max_dd', 'exits'], got {list(sweep.columns)}"
assert list(sweep.index) == [0.10, 0.15, 0.20, 0.25, 0.30], "one row per width, in order"
assert sweep["total_return"].dtype.kind == "f", "total_return must be numeric, not a formatted string"
assert (sweep["max_dd"] <= 0).all(), "drawdowns are negative numbers"
# Recompute one row independently.
_pos = trailing_stop(prices["TECH"], 0.20).shift(1).fillna(1)
_net = (_pos * rets["TECH"]).dropna()
_eq = (1 + _net).cumprod()
assert np.isclose(sweep.loc[0.20, "total_return"], float(_eq.iloc[-1] - 1)), \
    "total_return is off for the 20% width"
assert sweep.loc[0.20, "exits"] == int((_pos.diff() == -1).sum()), "exits count is off"
# Wider stops must trade less.
assert sweep.loc[0.10, "exits"] > sweep.loc[0.30, "exits"], \
    "a tighter stop must fire more often than a wider one"
assert stable is False, (
    "the sweep should come back UNSTABLE - the spread of total returns across widths is "
    "far wider than 25 percentage points, which is the whole lesson of this section.")
print(f"✅ Correct!  Total return ranges {sweep['total_return'].min():.0%} to",
      f"{sweep['total_return'].max():.0%} across five widths -> stable: {stable}.")
print(f"   Drawdown ranges {sweep['max_dd'].max():.1%} to {sweep['max_dd'].min():.1%},",
      "and neither column moves monotonically with the width.")
print("   A rule this parameter-sensitive has not been validated by its backtest.")

✅ Correct!  Total return ranges 109% to 251% across five widths -> stable: False.
   Drawdown ranges -36.3% to -55.7%, and neither column moves monotonically with the width.
   A rule this parameter-sensitive has not been validated by its backtest.


## 12. Leverage: the one that ends accounts

Module 6 section 1 mentioned the leverage trap in passing. Here is the arithmetic.

Leverage multiplies returns **and** volatility linearly, so it leaves Sharpe unchanged — but it multiplies drawdown too, and drawdown compounds against you. At 3× leverage, a 34% underlying decline is a total loss, and you will be liquidated well before that because margin requirements bite first.

In [25]:
base_dd = abs(float(((1 + port).cumprod()
                     / (1 + port).cumprod().cummax() - 1).min()))

rows = []
for lev in [1.0, 1.5, 2.0, 3.0]:
    levered = port * lev
    eq = (1 + levered).cumprod()
    dd = float((eq / eq.cummax() - 1).min())
    rows.append({
        "leverage": f"{lev:.1f}x",
        "total": f"{eq.iloc[-1] - 1:.1%}",
        "ann_vol": f"{levered.std() * np.sqrt(252):.1%}",
        "sharpe": round(float((levered.mean() * 252) / (levered.std() * np.sqrt(252))), 2),
        "max_dd": f"{dd:.1%}",
        "wipeout_at": f"{1 / lev:.0%} decline",
    })

print(pd.DataFrame(rows).to_string(index=False))

leverage  total ann_vol  sharpe max_dd   wipeout_at
    1.0x  92.4%   14.1%    0.65 -48.4% 100% decline
    1.5x 151.4%   21.1%    0.65 -63.5%  67% decline
    2.0x 215.5%   28.1%    0.65 -74.5%  50% decline
    3.0x 339.6%   42.2%    0.65 -87.9%  33% decline


Sharpe is identical down the column — leverage adds no skill. Drawdown, meanwhile, goes from −48% to −87%, and at 3× the account is mathematically destroyed by a 33% fall in the underlying.

The `wipeout_at` column assumes you are allowed to ride it down. You are not: a margin call liquidates you at the worst possible moment, turning a temporary loss into a permanent one.

> ⚠️ **Leverage is not a way to improve a strategy. It is a way to convert an existing edge into a larger position and a larger risk of ruin.** If a strategy is not worth trading at 1×, leverage does not fix it — it just gets you to zero faster.

On QuantConnect, `set_holdings("SPY", 2.0)` requests 2× leverage. Whether you get it depends on the brokerage model and the security's margin requirements, which is one more reason to set a realistic `set_brokerage_model` (Module 6 section 1) rather than the permissive default.

## Part 3 — Competition readiness

## 13. Putting the risk controls together

Now stack them **genuinely cumulatively** — each row is the row above it plus one more control, all measured from the same start date so the comparison is like for like.

Note the detail in row 3: the volatility target is computed from the *inverse-vol book's* own realised volatility, not from the equal-weight portfolio's. Section 9 targeted the equal-weight book because that was its baseline. Layering means re-estimating on what you are actually holding — carrying section 9's `scale` over unchanged would size row 3 using a portfolio it no longer trades.

In [26]:
# Layer 3: vol-target the inverse-vol book using ITS OWN trailing volatility.
iv_realised = inv_vol_port.rolling(60).std() * np.sqrt(252)
iv_scale = (TARGET_VOL / iv_realised).shift(1).clip(upper=2.0)
layered_vt = iv_scale * inv_vol_port

# Layer 4: add the portfolio drawdown switch on top of that.
switch_pos = drawdown_switch(layered_vt.fillna(0.0), 0.20, 0.05)
layered_final = layered_vt.where(switch_pos != 0, 0.0)

print(pd.DataFrame([
    stats(port.loc[start:], "1. equal weight (baseline)"),
    stats(inv_vol_port.loc[start:], "2. + inverse-vol sizing"),
    stats(layered_vt.loc[start:], "3. + vol targeting (12%)"),
    stats(layered_final.loc[start:], "4. + 20% drawdown switch"),
]).to_string(index=False))
print(f"\nlayer 3 leverage: average {iv_scale.loc[start:].mean():.2f}, "
      f"range {iv_scale.loc[start:].min():.2f} to {iv_scale.loc[start:].max():.2f}")
print(f"layer 4 days flat: {int((layered_final.loc[start:] == 0).sum())}")

                  strategy  total ann_vol  sharpe max_dd
1. equal weight (baseline)  82.6%   14.1%    0.62 -48.4%
   2. + inverse-vol sizing  88.4%   11.8%    0.75 -36.9%
  3. + vol targeting (12%) 142.5%   12.5%    0.97 -26.5%
  4. + 20% drawdown switch 102.5%   11.8%    0.83 -24.8%

layer 3 leverage: average 1.10, range 0.49 to 2.00
layer 4 days flat: 276


Each layer does exactly what Part 2 said it would.

Rows 1 to 3 improve **every** column: return 82.6% → 142.5%, volatility 14.1% → 12.5%, Sharpe 0.62 → 0.97, drawdown −48.4% → −26.5%. Those two controls are close to free.

Row 4 is the trade. The drawdown switch takes 40 points of return and 0.14 of Sharpe, and hands back 1.7 points of drawdown. On this data that is a poor exchange, because the vol targeting in row 3 had already removed most of the crash exposure the switch exists to remove — by the time the switch fires, the position it flattens is a small one.

> 🧠 **Risk controls overlap.** Adding a second control on top of a first that already handles the same scenario costs you its full premium and buys you a fraction of its benefit. Measure each layer against the layer below it, not against the naked baseline, or you will stack four controls and conclude that all four were pulling their weight.

Which of rows 3 and 4 you submit is a judgement call about the mandate you trade under, not a question with one right answer. Say which you chose and why — that sentence is worth more to a reader than either set of numbers.

## 14. The competition checklist

Judges and reviewers converge on the same questions. Most entries fail on the first four, which are about honesty rather than performance.

### Correctness — get these wrong and nothing else counts

- [ ] **No look-ahead.** Every signal uses `shift(1)` or an equivalent guard. (Module 5 §8)
- [ ] **No survivorship bias.** The universe is as-of each date, not today's index members. (Module 6 §2)
- [ ] **Costs and slippage modelled**, with a realistic brokerage model. (Module 6 §1, §6)
- [ ] **Warm-up set**, so no indicator trades before it is ready. (Module 5 §4)
- [ ] **Out-of-sample period held back** and reported separately. (Module 6 §8.1)
- [ ] **ML pipelines pass the scrambled-target test.** (Module 8 §12)

### Robustness — is the result a property of the strategy or of the sample?

- [ ] Performance reported per year, not only in aggregate. (Module 6 §8.2)
- [ ] Parameters swept, with the sensitivity shown. (Module 6 §8.3)
- [ ] Number of strategies searched disclosed. (Module 6 §8.4)
- [ ] Enough trades to be statistically meaningful. (Module 6 §8.5)
- [ ] Breakeven cost stated, not just gross return. (Module 8 §10)

### Risk — can this actually be run?

- [ ] Maximum drawdown within the mandate.
- [ ] Position and sector concentration limits in place.
- [ ] Leverage stated and justified.
- [ ] Behaviour in the worst period described, not hidden.

### Presentation

- [ ] The economic rationale stated in one sentence, before any statistics.
- [ ] The strategy's failure mode named explicitly.
- [ ] Code that runs from a clean clone.

> 🧠 **Volunteering a weakness is a strength.** A reviewer who finds a flaw you did not mention discounts everything else you claimed. A reviewer who reads *"this is short volatility and will lose badly in a gap-down; here is the sizing that keeps that survivable"* trusts the rest of the report.

## 15. What actually disqualifies entries

Four failure modes, in the order they occur.

**Look-ahead bias.** A Sharpe above about 3 on daily equity data is a bug until proven otherwise. Find it by shifting every signal one more day: a real edge degrades a little, a look-ahead bug collapses.

**Overfitting.** Fifty backtests and the best one submitted, with no mention of the other forty-nine. Modules 6, 7 and 8 each measured a version of this.

**Unrealistic fills.** Market-on-open fills at the open price, in size, on an illiquid symbol. Check that your assumed volume is a small fraction of the actual.

**Untested risk.** A strategy whose worst historical drawdown falls outside the period tested. Run it through 2008, 2020 and 2022 before believing anything.

### ✏️ Your turn — audit an algorithm

Write `readiness_audit(config)` that takes a dict describing an algorithm and returns `(score, failures)` where `score` is the number of checks passed as an int and `failures` is a **sorted list** of the names of the checks that failed.

Apply these five checks, using exactly these names:

| name | passes when |
|---|---|
| `"no_lookahead"` | `config["signals_shifted"]` is `True` |
| `"costs_modelled"` | `config["cost_bps"]` is greater than `0` |
| `"out_of_sample"` | `config["oos_years"]` is at least `1` |
| `"enough_trades"` | `config["n_trades"]` is at least `30` |
| `"drawdown_ok"` | `abs(config["max_dd"])` is at most `config["dd_limit"]` |

Then audit `good_config` and `bad_config` (both defined in the stub) as `(good_score, good_fails)` and `(bad_score, bad_fails)`.

In [27]:
good_config = {"signals_shifted": True, "cost_bps": 5, "oos_years": 2,
               "n_trades": 145, "max_dd": -0.18, "dd_limit": 0.25}

bad_config = {"signals_shifted": False, "cost_bps": 0, "oos_years": 0,
              "n_trades": 12, "max_dd": -0.52, "dd_limit": 0.25}


def readiness_audit(config):
    checks = {
        "no_lookahead": config["signals_shifted"] is True,
        "costs_modelled": config["cost_bps"] > 0,
        "out_of_sample": config["oos_years"] >= 1,
        "enough_trades": config["n_trades"] >= 30,
        "drawdown_ok": abs(config["max_dd"]) <= config["dd_limit"],
    }
    score = int(sum(checks.values()))
    failures = sorted(name for name, ok in checks.items() if not ok)
    return score, failures


good_score, good_fails = readiness_audit(good_config)
bad_score, bad_fails = readiness_audit(bad_config)

print(f"good: {good_score}/5  failures: {good_fails}")
print(f"bad : {bad_score}/5  failures: {bad_fails}")

good: 5/5  failures: []
bad : 0/5  failures: ['costs_modelled', 'drawdown_ok', 'enough_trades', 'no_lookahead', 'out_of_sample']


In [28]:
assert good_score == 5, f"good_config should pass all five, got {good_score}"
assert good_fails == [], f"good_config should have no failures, got {good_fails}"
assert bad_score == 0, f"bad_config should fail all five, got {bad_score}"
assert bad_fails == ["costs_modelled", "drawdown_ok", "enough_trades",
                     "no_lookahead", "out_of_sample"], \
    f"failures must be SORTED names of the failed checks, got {bad_fails}"
assert isinstance(good_score, int), "score must be an int"
# Each check must be independent - flip one field at a time.
for field, value, expect in [("signals_shifted", False, "no_lookahead"),
                             ("cost_bps", 0, "costs_modelled"),
                             ("oos_years", 0, "out_of_sample"),
                             ("n_trades", 29, "enough_trades"),
                             ("max_dd", -0.99, "drawdown_ok")]:
    _c = dict(good_config)
    _c[field] = value
    _s, _f = readiness_audit(_c)
    assert _f == [expect], f"flipping '{field}' should fail exactly ['{expect}'], got {_f}"
# Boundaries are inclusive where the table says "at least" / "at most".
_b = dict(good_config); _b["n_trades"] = 30
assert readiness_audit(_b)[0] == 5, "exactly 30 trades should PASS (at least 30)"
_b = dict(good_config); _b["max_dd"] = -0.25
assert readiness_audit(_b)[0] == 5, "a drawdown exactly at the limit should PASS (at most)"
print(f"✅ Correct!  {good_score}/5 and {bad_score}/5.",
      "Run this against your own entry before you submit it, and be honest with the inputs -",
      "the audit is only as good as the config you hand it.")

✅ Correct!  5/5 and 0/5. Run this against your own entry before you submit it, and be honest with the inputs - the audit is only as good as the config you hand it.


## 16. The capstone algorithm

Everything from Modules 3 through 9 in one file: a framework algorithm with a real universe, an alpha model, volatility-based sizing, layered risk management, and honest cost assumptions.

In [ ]:
# 🔵 QC cell — the capstone
from AlgorithmImports import *
from datetime import timedelta
import numpy as np


class CompetitionEntryAlgorithm(QCAlgorithm):

    def initialize(self):
        # --- period, capital, and a realistic broker (Module 6 s1) --------------
        self.set_start_date(2016, 1, 1)
        self.set_end_date(2022, 1, 1)          # 2022+ held back as out-of-sample
        self.set_cash(100_000)
        self.set_brokerage_model(BrokerageName.INTERACTIVE_BROKERS_BROKERAGE,
                                 AccountType.MARGIN)
        self.settings.free_portfolio_value_percentage = 0.05

        # --- warm-up, so nothing trades on an unready indicator (Module 5 s4) ---
        self.set_warm_up(120, Resolution.DAILY)

        universe = [Symbol.create(t, SecurityType.EQUITY, Market.USA)
                    for t in ["SPY", "QQQ", "XLF", "XLP", "GLD", "TLT"]]

        self.set_universe_selection(ManualUniverseSelectionModel(universe))
        self.set_alpha(MaCrossAlphaModel(fast=20, slow=60, period_days=5))
        self.set_portfolio_construction(InverseVolatilityPortfolioConstructionModel(60))

        # --- layered risk: per-position first, then portfolio-level ------------
        self.add_risk_management(MaximumDrawdownPercentPerSecurity(0.10))
        self.add_risk_management(PortfolioDrawdownRiskModel(0.20, 0.05))

        self.set_execution(ImmediateExecutionModel())

        # --- a diagnostic worth plotting (Module 6 s9) -------------------------
        self.schedule.on(self.date_rules.month_start(),
                         self.time_rules.at(16, 0),
                         self.record_exposure)

    def record_exposure(self):
        if self.is_warming_up:
            return
        gross = sum(abs(x.holdings_value) for x in self.portfolio.values())
        self.plot("Exposure", "gross", gross / self.portfolio.total_portfolio_value)
        self.plot("Exposure", "cash", self.portfolio.cash / self.portfolio.total_portfolio_value)

Run it, then answer these before showing it to anyone.

1. What is the **economic reason** this should work? If the only answer is "the backtest is good", you have a curve fit.
2. What does the **worst year** look like, not the average?
3. How many variants did you try before this one?
4. What is the **breakeven cost**, and is it realistic for these symbols?
5. What **market condition** breaks it? Name it specifically.

Question 5 is the one that separates a submission from a report. Every strategy has an answer; not knowing it means you have not looked.

## Cheat sheet

**Framework wiring**

| Task | Code |
|---|---|
| Universe | `self.set_universe_selection(ManualUniverseSelectionModel(symbols))` |
| Alpha | `self.set_alpha(MyAlphaModel())` |
| Sizing | `self.set_portfolio_construction(EqualWeightingPortfolioConstructionModel())` |
| Risk | `self.add_risk_management(MaximumDrawdownPercentPortfolio(0.10))` |
| Execution | `self.set_execution(ImmediateExecutionModel())` |
| Emit a view | `Insight.price(symbol, timedelta(days=5), InsightDirection.UP)` |
| Request a size | `PortfolioTarget.percent(algorithm, symbol, 0.25)` |
| Close a position | `PortfolioTarget(symbol, 0)` |
| Build indicators | in `on_securities_changed`, **not** `initialize` |

**Risk arithmetic**

| Quantity | Formula |
|---|---|
| Gain needed to recover | `1 / (1 - dd) - 1` |
| Portfolio volatility | `sqrt(w @ cov @ w)` |
| Diversification ratio | `(w @ individual_vols) / portfolio_vol` |
| Inverse-vol weights | `(1/σ) / Σ(1/σ)`, then `.shift(1)` |
| Vol-target leverage | `target_vol / trailing_vol`, then `.shift(1)`, then clip |
| Max drawdown | `(eq / eq.cummax() - 1).min()` |

**What each technique actually costs**

| Technique | Return | Drawdown | Sharpe | Needs tuning? |
|---|---|---|---|---|
| Diversification | — | much lower | higher | no |
| Inverse-vol sizing | higher | lower | higher | barely |
| Vol targeting | higher | lower | higher | one target |
| Drawdown switch | **lower** | much lower | slightly lower | yes — dangerous |
| Trailing stop | unpredictable | unpredictable | unpredictable | yes — very dangerous |
| Leverage | scales | **scales** | unchanged | n/a |

## Stretch goals

1. **Swap one component.** Take the capstone and change only the portfolio construction model to `EqualWeightingPortfolioConstructionModel`. Report the difference in Sharpe and drawdown. This is the Framework's whole value proposition — verify that it is real.
2. **Write a risk model that scales rather than flattens.** Instead of going to cash at the drawdown limit, halve every target. Compare against the binary switch in section 10. Does a gentler response keep more of the return?
3. **Rebalance bands.** Section 9 noted that daily vol-target rebalancing generates turnover. Only adjust the scale when it drifts more than 20% from its current value, charge 10bp per unit of turnover, and see how much of the improvement survives.
4. **Correlation in a crisis.** Compute the correlation matrix separately inside and outside the crash window. Correlations tend toward 1 exactly when you need diversification most. Quantify it here, then re-run section 7's diversification ratio using crisis-only data.
5. **Run the whole audit.** Take your best strategy from any module and complete section 14's checklist honestly. Write down every item you cannot tick.
6. **The out-of-sample test.** The capstone ends in 2022. Run it forward to today, on data you have never looked at, and compare. Whatever the result, that number is the most honest one you have produced in this course.

## What's next

You have finished the curriculum. Across nine modules you have gone from NumPy arrays to a framework algorithm with layered risk management, and the through-line was never a particular technique.

It was this: **the hard part of quantitative trading is not finding a pattern, it is establishing that a pattern you found is real.** Module 6 searched 200 strategies and found a great one in noise. Module 7 screened 1,770 pairs and found dozens of cointegrated relationships in data containing none. Module 8 produced a 72% classifier that knew nothing. Each of those results looked exactly like success.

The tools that separate the real from the imaginary — out-of-sample testing, walk-forward validation, cost sensitivity, parameter stability, the scrambled-target test — are unglamorous and they are the entire job.

**Where to go from here:**
- Enter a competition. The [QuantConnect Quant League](https://www.quantconnect.com/competitions) runs regularly.
- Paper trade something for three months. Nothing else teaches the gap between backtest and live.
- Read *Advances in Financial Machine Learning* (Marcos López de Prado) for the rigorous treatment of Module 8's purging and embargo.
- Bring a strategy to a MAT meeting and let people attack it. Free adversarial review is the most valuable thing a club offers.

**Official docs:**
- [QuantConnect: Algorithm Framework](https://www.quantconnect.com/docs/v2/writing-algorithms/algorithm-framework/overview)
- [QuantConnect: Risk Management models](https://www.quantconnect.com/docs/v2/writing-algorithms/algorithm-framework/risk-management/supported-models)
- [QuantConnect: Portfolio Construction models](https://www.quantconnect.com/docs/v2/writing-algorithms/algorithm-framework/portfolio-construction/supported-models)

*MAT Education · ML & Capstone · Module 9. End of series.*